![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Featureform Transformations & Feature Lineage

In this recipe we build features with **chained SQL transformations** in [**Featureform**](https://docs.featureform.com/), and follow the **lineage** Featureform records from raw data all the way to a served feature.

## Why chained transformations and lineage matter
A production feature is rarely a raw column — it's the result of a *pipeline*: clean the raw events, then aggregate them. Two things go wrong without a feature store:
- **Nobody can tell how a value was produced.** When a model misbehaves you need to trace a feature back through every step to its source. Ad-hoc scripts don't record that path.
- **Steps get duplicated and drift.** The "clean transactions" step gets rewritten slightly differently by every downstream job.

Featureform fixes both by making every transformation a **named, versioned resource** whose inputs are other named resources. That dependency graph *is* the lineage: `raw source → clean → aggregate → feature`, recorded and visualizable, with each step defined exactly once.

## What we'll build
A two-step feature pipeline over a transactions dataset:
1. **`clean_transactions`** — a transformation that filters out invalid rows from the raw source.
2. **`avg_user_transaction`** — a second transformation that reads the *output of step 1* (via the `{{clean_transactions.quickstart}}` reference) and aggregates per user. That reference is what creates the lineage edge.

We then define an `avg_transaction_amt` feature from step 2, register a training set, `apply()` everything, and walk each stage — seeing the exact DataFrame Featureform computed at every node in the graph.

## The stack — no Spark, no cloud

Featureform separates **compute/offline** (where transformations run) from the **online store** (where features are served). Transformations here are **SQL**, which run directly in a SQL offline store — so there's no Spark cluster and no object storage to stand up. This recipe uses:
- **ClickHouse** as the offline store — a columnar SQL database that runs the transformations. One local Docker container.
- **Redis** as the online store, for low-latency serving of the finished feature.

> ℹ️ **Transformations are SQL, not pandas here.** Featureform's pandas (`df_transformation`) support requires a Spark or Kubernetes provider. SQL transformations cover the same clean → aggregate → serve pipeline with none of that infrastructure.

> ⚠️ **This notebook needs local Docker and will not run on Colab or in CI.** You need three things running:
> 1. A **Featureform** coordinator (gRPC on `localhost:7878`, dashboard on `http://localhost`) — see the [Featureform install docs](https://docs.featureform.com/deployment/quickstart-docker).
> 2. A **ClickHouse** container (below).
> 3. A **Redis** container (below).

### Start ClickHouse and Redis

Skip any container you already have running. ClickHouse exposes its HTTP port `8123` (used below to load data) and its native port `9000` (used by the Featureform coordinator).

In [ ]:
# NBVAL_SKIP
!docker run -d --name clickhouse -p 8123:8123 -p 9000:9000 clickhouse/clickhouse-server:latest
!docker run -d --name redis -p 6379:6379 redis:8

## Environment Setup

### Install Python Dependencies

In [ ]:
%pip install -q featureform redis clickhouse-connect pandas

### Configure connections

Connection values are driven by environment variables so you can point the notebook at your own instances. `host.docker.internal` is how the Featureform coordinator *container* reaches ClickHouse and Redis published on your host (on Linux, use the Docker bridge IP `172.17.0.1`).

In [ ]:
import os

# Featureform coordinator (gRPC)
FEATUREFORM_HOST = os.getenv("FEATUREFORM_HOST", "localhost:7878")

# Address the coordinator container uses to reach the providers.
PROVIDER_HOST = os.getenv("PROVIDER_HOST", "host.docker.internal")

# ClickHouse offline store (default user, no password)
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", PROVIDER_HOST)
CLICKHOUSE_NATIVE_PORT = int(os.getenv("CLICKHOUSE_NATIVE_PORT", "9000"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "default")

# Redis online store
REDIS_HOST = os.getenv("REDIS_HOST", PROVIDER_HOST)
REDIS_PORT = int(os.getenv("REDIS_PORT", "6379"))
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

### Create a sample table in ClickHouse

So the notebook is self-contained, we create a `transactions` table and load a small, intentionally *messy* dataset (some invalid negative amounts) so the first transformation has something to clean. In a real deployment this table would already exist. We connect over ClickHouse's HTTP port `8123` from here.

In [ ]:
# NBVAL_SKIP
import clickhouse_connect
import numpy as np

ch = clickhouse_connect.get_client(
    host="localhost", port=8123,
    username=CLICKHOUSE_USER, password=CLICKHOUSE_PASSWORD,
)

ch.command("DROP TABLE IF EXISTS transactions")
ch.command(
    """
    CREATE TABLE transactions (
        TransactionID String,
        CustomerID String,
        TransactionAmount Float64,
        IsFraud Bool
    ) ENGINE = MergeTree ORDER BY CustomerID
    """
)

rng = np.random.default_rng(42)
n = 500
rows = []
for i in range(n):
    amount = round(float(rng.gamma(2.0, 50.0)), 2)
    if i % 150 == 0:
        amount = -1.0   # a few invalid amounts for the cleaning step to drop
    rows.append([
        f"T{i:05d}",
        f"C{int(rng.integers(1000, 1050)):04d}",
        amount,
        bool(rng.integers(0, 2)),
    ])

ch.insert("transactions", rows,
          column_names=["TransactionID", "CustomerID", "TransactionAmount", "IsFraud"])
print("rows in ClickHouse:", ch.command("SELECT count() FROM transactions"))

## Register the providers

We register the **ClickHouse** offline store and the **Redis** online store. Registering a provider just tells Featureform how to reach it — no data moves yet.

In [ ]:
import featureform as ff

clickhouse = ff.register_clickhouse(
    name="clickhouse-quickstart",
    description="ClickHouse offline store (runs the SQL transformations)",
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_NATIVE_PORT,
    user=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

redis = ff.register_redis(
    name="redis-quickstart",
    description="Redis online (inference) store",
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=0,
)

## Register the raw source

We point Featureform at the ClickHouse `transactions` table. This becomes the **root node** of our lineage graph — the source that transformations build on.

In [ ]:
transactions = clickhouse.register_table(
    name="transactions",
    variant="quickstart",
    table="transactions",  # the table name in ClickHouse
)

## Step 1 — a cleaning transformation

A `sql_transformation` is a plain function that returns a SQL string. The `{{transactions.quickstart}}` placeholder references the source we just registered. This query runs **in ClickHouse**, and its result becomes a new, named source. Here we drop the invalid (non-positive) amounts.

In [ ]:
@clickhouse.sql_transformation(variant="quickstart")
def clean_transactions():
    """Keep only valid transactions."""
    return (
        "SELECT CustomerID, TransactionAmount, IsFraud "
        "FROM {{transactions.quickstart}} WHERE TransactionAmount > 0"
    )

## Step 2 — an aggregation that *chains* on Step 1

This is the key moment for lineage: the `FROM` clause references **`{{clean_transactions.quickstart}}`, not the raw source**. Featureform records the edge `clean_transactions → avg_user_transaction`, so the dependency is explicit and traceable — and Step 1's logic is defined once and reused, never copy-pasted.

We aggregate each user's cleaned transactions into an average amount and a count.

In [ ]:
@clickhouse.sql_transformation(variant="quickstart")
def avg_user_transaction():
    """Average transaction amount and count per user, from the *cleaned* data."""
    return (
        "SELECT CustomerID AS user_id, "
        "avg(TransactionAmount) AS avg_transaction_amt, "
        "count(*) AS transaction_count "
        "FROM {{clean_transactions.quickstart}} GROUP BY CustomerID"
    )

## Define the entity, feature, and label

`@ff.entity` groups resources keyed by a **user**. The feature is sourced from the *Step 2* transformation and materialized to Redis for serving. The label (`IsFraud`) comes from the *Step 1* transformation and stays offline — it's only used to build training sets.

Notice the feature and label draw from **different nodes of the same lineage graph**, both ultimately rooted in the raw source.

In [ ]:
@ff.entity
class User:
    avg_transactions = ff.Feature(
        avg_user_transaction[["user_id", "avg_transaction_amt"]],
        variant="quickstart",
        type=ff.Float32,
        inference_store=redis,
    )
    fraudulent = ff.Label(
        clean_transactions[["CustomerID", "IsFraud"]],
        variant="quickstart",
        type=ff.Bool,
    )

## Register a training set

A training set joins the feature(s) to the label on the entity key — built from the same definitions that serve online, so there's no training-serving skew.

In [ ]:
ff.register_training_set(
    "fraud_training",
    variant="quickstart",
    label=("fraudulent", "quickstart"),
    features=[("avg_transactions", "quickstart")],
)

## Apply the definitions

`client.apply()` sends everything to the coordinator and runs the pipeline: ClickHouse executes `clean_transactions`, then `avg_user_transaction` on its output, then the feature is materialized into Redis. `asynchronous=False` blocks until it finishes.

In [ ]:
# NBVAL_SKIP
client = ff.Client(host=FEATUREFORM_HOST, insecure=True)
client.apply(asynchronous=False, verbose=True)

## Follow the lineage stage by stage

`client.dataframe()` computes and returns the DataFrame at any node in the graph. Reading them in order — raw → cleaned → aggregated — you can *see* the pipeline that produced the feature, each step traceable to the one before it. This is the lineage, made concrete.

In [ ]:
# NBVAL_SKIP
print("1. Raw source:")
display(client.dataframe(transactions).head())

print("2. After clean_transactions (invalid amounts gone):")
display(client.dataframe(clean_transactions).head())

print("3. After avg_user_transaction (aggregated per user):")
display(client.dataframe(avg_user_transaction).head())

### See the lineage graph visually

The Featureform dashboard at **http://localhost** renders the dependency DAG for every resource. Open the `avg_transactions` feature and you'll see the chain `transactions → clean_transactions → avg_user_transaction → avg_transactions` — the same lineage you just walked in code, plus variants, owners, and timestamps for audit.

## Serve the finished feature from Redis

The end of the pipeline: request the feature for a single entity key. This read is served from Redis at low latency — the value having flowed through the whole traceable pipeline to get here. We grab a `user_id` that appears in the aggregated output above.

In [ ]:
# NBVAL_SKIP
user_id = client.dataframe(avg_user_transaction)["user_id"].iloc[0]
avg_txn = client.features(
    [("avg_transactions", "quickstart")],
    {"user": user_id},
)
print(f"avg_transactions for user {user_id}:", avg_txn)

## Build a training set from the same definitions

The offline side reuses the exact feature/label definitions. The dataset is iterable and streams rows of `(features, label)`.

In [ ]:
# NBVAL_SKIP
dataset = client.training_set("fraud_training", "quickstart")

for i, row in enumerate(dataset):
    print(row.features(), "->", row.label())
    if i >= 4:
        break

## Cleanup

Stop and remove the containers when you're done.

In [ ]:
# NBVAL_SKIP
!docker rm -f clickhouse redis

## Learn more

- [Featureform transformations](https://docs.featureform.com/) — SQL and DataFrame transformations, chaining, and variants
- [Featureform + Redis fraud detection recipe](./02_featureform_fraud_detection.ipynb) — a single-transformation version of this pattern